# Titanic EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
train.head()

## 基本統計量

In [ ]:
train.describe()

## 欠損値

In [ ]:
missing = train.isnull().sum()
missing[missing > 0].sort_values(ascending=False).plot.bar()
plt.title('Missing Values')
plt.ylabel('Count')
plt.show()

## 生存率

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.countplot(x='Survived', data=train, ax=axes[0])
axes[0].set_title('Survived')

sns.countplot(x='Survived', hue='Sex', data=train, ax=axes[1])
axes[1].set_title('Survived by Sex')

sns.countplot(x='Survived', hue='Pclass', data=train, ax=axes[2])
axes[2].set_title('Survived by Pclass')

plt.tight_layout()
plt.show()

## 年齢ごとの人数

In [ ]:
plt.figure(figsize=(8, 5))
bins = range(0, 90, 10)
train['Age'].dropna().plot.hist(bins=bins, edgecolor='black', rwidth=0.9)
plt.xticks(bins)
plt.title('Number of Passengers by Age (10-year bins)')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 年齢(10歳刻み) x 性別ごとの生存率

In [ ]:
train['AgeGroup'] = pd.cut(train['Age'], bins=range(0, 90, 10), right=False,
                          labels=[f'{i}-{i+9}' for i in range(0, 80, 10)])
survival_table = train.groupby(['AgeGroup', 'Sex'])['Survived'].mean().unstack()
survival_table.style.format('{:.1%}').background_gradient(cmap='RdYlGn', vmin=0, vmax=1)

## 運賃分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train['Fare'].hist(bins=30, ax=axes[0])
axes[0].set_title('Fare Distribution')

sns.boxplot(x='Pclass', y='Fare', data=train, ax=axes[1])
axes[1].set_title('Fare by Pclass')

plt.tight_layout()
plt.show()

## 相関行列

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(train.select_dtypes(include='number').corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## 家族生存戦略の階層別分析

仮説: **3等の大家族は共倒れ（全員死亡）し、1等の大家族は子供を優先救出した**

### Task A: FamilySize × Pclass 生存率曲線

FamilySizeごとの生存率を Pclass別に折れ線グラフで描画。5人以上の家族で3等の生存率が急落するか確認。

In [ ]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: Pclass別 FamilySize vs 生存率
for pclass in [1, 2, 3]:
    subset = train[train['Pclass'] == pclass]
    survival = subset.groupby('FamilySize')['Survived'].agg(['mean', 'count'])
    axes[0].plot(survival.index, survival['mean'], marker='o', label=f'Pclass {pclass}')
    # サンプル数をアノテーション
    for fs, row in survival.iterrows():
        axes[0].annotate(f'n={int(row["count"])}', (fs, row['mean']),
                        textcoords='offset points', xytext=(0, 8), fontsize=7, ha='center')

axes[0].set_xlabel('FamilySize')
axes[0].set_ylabel('Survival Rate')
axes[0].set_title('Survival Rate by FamilySize & Pclass')
axes[0].legend()
axes[0].set_xticks(range(1, train['FamilySize'].max() + 1))
axes[0].set_ylim(-0.05, 1.05)
axes[0].axhline(y=train['Survived'].mean(), color='gray', linestyle='--', alpha=0.5, label='Overall avg')

# 右: 全体の FamilySize vs 生存率 (参考)
survival_all = train.groupby('FamilySize')['Survived'].agg(['mean', 'count'])
axes[1].bar(survival_all.index, survival_all['count'], color='steelblue', alpha=0.6, label='Count')
ax2 = axes[1].twinx()
ax2.plot(survival_all.index, survival_all['mean'], color='red', marker='s', linewidth=2, label='Survival Rate')
axes[1].set_xlabel('FamilySize')
axes[1].set_ylabel('Count')
ax2.set_ylabel('Survival Rate')
axes[1].set_title('FamilySize Distribution & Survival Rate')
axes[1].set_xticks(range(1, train['FamilySize'].max() + 1))
ax2.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

# 数値テーブル
pivot = train.groupby(['Pclass', 'FamilySize'])['Survived'].agg(['mean', 'count']).unstack(level=0)
pivot.columns = [f'Pclass{c[1]}_{c[0]}' for c in pivot.columns]
pivot

### Task B: All-or-Nothing 分析

FamilySize>1 の家族グループ（Surname + Ticket でグルーピング）を作り、各グループの生存パターンを分類:
- **全員生存**: グループ全員 Survived=1
- **全員死亡**: グループ全員 Survived=0
- **混合**: 一部生存・一部死亡

Pclass別にこの割合を比較。3等の大家族は「全員死亡」が多いか？

In [ ]:
# 家族グループを Surname で定義（FamilySize > 1 のみ）
train['Surname'] = train['Name'].str.split(',').str[0]

family = train[train['FamilySize'] > 1].copy()

# Surname + Pclass でグルーピング（同姓でもクラスが違えば別家族の可能性）
family['FamilyGroup'] = family['Surname'] + '_' + family['Pclass'].astype(str)

# 各家族グループの生存パターンを判定
family_stats = family.groupby('FamilyGroup').agg(
    size=('Survived', 'size'),
    survived_sum=('Survived', 'sum'),
    pclass=('Pclass', 'first'),
    family_size=('FamilySize', 'first')
).reset_index()

def classify_pattern(row):
    if row['survived_sum'] == 0:
        return 'All Died'
    elif row['survived_sum'] == row['size']:
        return 'All Survived'
    else:
        return 'Mixed'

family_stats['pattern'] = family_stats.apply(classify_pattern, axis=1)

# Pclass別の分布
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, pclass in enumerate([1, 2, 3]):
    subset = family_stats[family_stats['pclass'] == pclass]
    counts = subset['pattern'].value_counts()
    total = len(subset)
    colors = {'All Survived': '#2ecc71', 'Mixed': '#f39c12', 'All Died': '#e74c3c'}
    bars = axes[i].bar(counts.index, counts.values, 
                       color=[colors.get(x, 'gray') for x in counts.index])
    axes[i].set_title(f'Pclass {pclass} (n={total} families)')
    axes[i].set_ylabel('Number of Family Groups')
    # パーセンテージ表示
    for bar, val in zip(bars, counts.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{val}\n({val/total:.0%})', ha='center', fontsize=10)

plt.suptitle('Family Survival Patterns by Pclass (FamilySize > 1)', fontsize=14)
plt.tight_layout()
plt.show()

# 大家族 (FamilySize >= 5) に絞った分析
print("=== 大家族 (FamilySize >= 5) のパターン ===")
large_family = family_stats[family_stats['family_size'] >= 5]
for pclass in [1, 2, 3]:
    subset = large_family[large_family['pclass'] == pclass]
    if len(subset) == 0:
        print(f"\nPclass {pclass}: 該当なし")
        continue
    print(f"\nPclass {pclass}: {len(subset)} families")
    print(subset[['FamilyGroup', 'size', 'survived_sum', 'pattern']].to_string(index=False))

### Task C: 子供 (Age < 16) の生存率比較

1等の大家族の子供 vs 3等の大家族の子供の生存率を比較。
1等では「子供を優先救出」パターンがあるなら、子供の生存率が高いはず。

In [ ]:
# 子供の定義: Age < 16
train['IsChild'] = (train['Age'] < 16).astype(int)

# FamilySize カテゴリ
train['FamilySizeCat'] = pd.cut(train['FamilySize'], bins=[0, 1, 3, 100], 
                                 labels=['Solo', 'Small(2-3)', 'Large(4+)'])

# 子供の生存率: Pclass × FamilySize
child_data = train[train['IsChild'] == 1].copy()

print(f"子供 (Age < 16) の総数: {len(child_data)}")
print()

# Pclass × FamilySizeCat ごとの子供の生存率
child_survival = child_data.groupby(['Pclass', 'FamilySizeCat']).agg(
    count=('Survived', 'size'),
    survived=('Survived', 'sum'),
    rate=('Survived', 'mean')
).reset_index()
print("=== 子供の生存率: Pclass × FamilySize ===")
print(child_survival.to_string(index=False))

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: Pclass別の子供 vs 大人の生存率
adult_child_survival = train.dropna(subset=['Age']).groupby(['Pclass', 'IsChild'])['Survived'].agg(['mean', 'count']).reset_index()
adult_child_survival['label'] = adult_child_survival['IsChild'].map({0: 'Adult (>=16)', 1: 'Child (<16)'})

for label, group in adult_child_survival.groupby('label'):
    axes[0].plot(group['Pclass'], group['mean'], marker='o', label=label, linewidth=2)
    for _, row in group.iterrows():
        axes[0].annotate(f'n={int(row["count"])}', (row['Pclass'], row['mean']),
                        textcoords='offset points', xytext=(5, 5), fontsize=8)

axes[0].set_xlabel('Pclass')
axes[0].set_ylabel('Survival Rate')
axes[0].set_title('Child vs Adult Survival Rate by Pclass')
axes[0].set_xticks([1, 2, 3])
axes[0].legend()
axes[0].set_ylim(-0.05, 1.05)

# 右: 大家族(4+)の子供 vs 小家族の子供 の生存率 by Pclass
for cat in ['Small(2-3)', 'Large(4+)']:
    subset = child_data[child_data['FamilySizeCat'] == cat]
    if len(subset) == 0:
        continue
    survival = subset.groupby('Pclass')['Survived'].agg(['mean', 'count'])
    axes[1].plot(survival.index, survival['mean'], marker='o', label=cat, linewidth=2)
    for pclass, row in survival.iterrows():
        axes[1].annotate(f'n={int(row["count"])}', (pclass, row['mean']),
                        textcoords='offset points', xytext=(5, 5), fontsize=8)

axes[1].set_xlabel('Pclass')
axes[1].set_ylabel('Survival Rate')
axes[1].set_title('Child Survival Rate: Small vs Large Family by Pclass')
axes[1].set_xticks([1, 2, 3])
axes[1].legend()
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

### まとめ・交互作用特徴量の提案

In [ ]:
# 仮説検証のまとめ数値
print("=" * 60)
print("仮説: 3等の大家族は共倒れ、1等の大家族は子供を優先救出")
print("=" * 60)

# 1. Pclass別の大家族(4+)生存率
print("\n【1】大家族 (FamilySize >= 4) の生存率 by Pclass")
large = train[train['FamilySize'] >= 4]
for pclass in [1, 2, 3]:
    sub = large[large['Pclass'] == pclass]
    if len(sub) > 0:
        print(f"  Pclass {pclass}: {sub['Survived'].mean():.1%} ({sub['Survived'].sum()}/{len(sub)})")

# 2. All-or-Nothing: 大家族の「全員死亡」率
print("\n【2】大家族グループの All-or-Nothing パターン")
large_groups = family_stats[family_stats['family_size'] >= 4]
for pclass in [1, 2, 3]:
    sub = large_groups[large_groups['pclass'] == pclass]
    if len(sub) > 0:
        all_died = (sub['pattern'] == 'All Died').sum()
        print(f"  Pclass {pclass}: 全員死亡 {all_died}/{len(sub)} ({all_died/len(sub):.0%}), "
              f"混合 {(sub['pattern'] == 'Mixed').sum()}/{len(sub)}")

# 3. 子供の生存率差
print("\n【3】子供 (Age < 16) の生存率: 大家族 vs 小家族")
children = train[(train['Age'] < 16) & (train['FamilySize'] > 1)].copy()
for pclass in [1, 2, 3]:
    sub = children[children['Pclass'] == pclass]
    if len(sub) == 0:
        continue
    small_kids = sub[sub['FamilySize'] <= 3]
    large_kids = sub[sub['FamilySize'] >= 4]
    print(f"  Pclass {pclass}:")
    if len(small_kids) > 0:
        print(f"    小家族(2-3): {small_kids['Survived'].mean():.1%} (n={len(small_kids)})")
    if len(large_kids) > 0:
        print(f"    大家族(4+) : {large_kids['Survived'].mean():.1%} (n={len(large_kids)})")

print("\n" + "=" * 60)
print("【提案する交互作用特徴量】")
print("  1. Pclass × FamilySize (or IsLargeFamily): 階級×家族規模の交互作用")
print("  2. Pclass × IsChild: 階級ごとの子供優遇度")
print("  3. Pclass × IsChild × IsLargeFamily: 3重交互作用")
print("  4. FamilySurvivalContext: 同一家族の他メンバーの生存情報 (リーク注意)")
print("=" * 60)